In [5]:

import os
import random
from pathlib import Path
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False
    print("⚠️ tqdm не установлен. Установите: pip install tqdm")

# =========================
# КОНФИГУРАЦИЯ
# =========================
BASE_DIR = Path(r"D:\lab3 product segmentation")
TRAIN_IMAGES = BASE_DIR / "1" / "images"
TRAIN_MASKS  = BASE_DIR / "1" / "masks"
SAVE_DIR     = BASE_DIR / "runs_improved"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 384
BATCH_SIZE = 10                     
NUM_EPOCHS = 40
LR = 1e-3
LR_MIN = 1e-6
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
THRESHOLD = 0.5
N_FOLDS = 5
USE_AMP = True
WARMUP_EPOCHS = 15

CUTMIX_PROB = 0.3
EMA_DECAY = 0.999

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Предзагрузка мощных энкодеров
print("🌐 Предзагрузка энкодеров...")
encoders_to_preload = [
    "resnet50", 
    "resnet101", 
    "efficientnet-b3", 
    "timm-efficientnet-b3",
    "efficientnet-b4"
]
for encoder in encoders_to_preload:
    try:
        _ = get_preprocessing_fn(encoder, pretrained="imagenet")
    except Exception as e:
        print(f"   ⚠️ {encoder} не загружен: {e}")
print("✅ Энкодеры готовы!\n")

# =========================
# УЛУЧШЕННАЯ ФУНКЦИЯ ПОТЕРЬ
# =========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()

class BoundaryLoss(nn.Module):
    def __init__(self):
        super().__init__()
        laplacian_kernel = torch.tensor([[[[0, 1, 0], [1, -4, 1], [0, 1, 0]]]], dtype=torch.float32)
        self.register_buffer('laplacian_kernel', laplacian_kernel)

    def forward(self, pred, target):
        pred_prob = torch.sigmoid(pred)
        kernel = self.laplacian_kernel.to(dtype=pred_prob.dtype, device=pred_prob.device)
        pred_boundary = torch.abs(F.conv2d(pred_prob, kernel, padding=1))
        target_boundary = torch.abs(F.conv2d(target, kernel, padding=1))
        return F.l1_loss(pred_boundary, target_boundary)

class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.4, focal_weight=0.3, boundary_weight=0.3):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.focal = FocalLoss()
        self.boundary = BoundaryLoss()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.boundary_weight = boundary_weight

    def forward(self, pred, target):
        loss = (self.dice_weight * self.dice(pred, target) +
                self.focal_weight * self.focal(pred, target) +
                self.boundary_weight * self.boundary(pred, target))
        return loss

# =========================
# CUTMIX
# =========================
def cutmix_segmentation(images, masks, alpha=0.5):
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    index = torch.randperm(batch_size).to(images.device)

    H, W = images.shape[2], images.shape[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    mixed_images = images.clone()
    mixed_masks = masks.clone()
    mixed_images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
    mixed_masks[:, :, y1:y2, x1:x2] = masks[index, :, y1:y2, x1:x2]

    return mixed_images, mixed_masks

# =========================
# EMA
# =========================
class EMAModel:
    def __init__(self, model, decay=EMA_DECAY):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.register()

    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

# =========================
# МЕТРИКИ
# =========================
def dice_score(pred, target, smooth=1e-7):
    pred = pred.contiguous().view(pred.size(0), -1)
    target = target.contiguous().view(target.size(0), -1)
    intersection = (pred * target).sum(dim=1)
    dice = (2. * intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)
    return dice.mean()

def iou_score(pred, target, smooth=1e-7):
    pred = pred.contiguous().view(pred.size(0), -1)
    target = target.contiguous().view(target.size(0), -1)
    intersection = (pred * target).sum(dim=1)
    union = pred.sum(dim=1) + target.sum(dim=1) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean()

# =========================
# ДАТАСЕТ
# =========================
class SegDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform
        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
                img_path = self.images_dir / f"{stem}{ext}"
                if img_path.exists():
                    self.samples.append((img_path, mask_path))
                    break
        print(f"Найдено {len(self.samples)} пар изображение-маска")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug['image']
            mask = aug['mask']
        else:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img.transpose(2, 0, 1)).float()
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
            mask = torch.from_numpy(mask).float().unsqueeze(0)
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        return img.float(), mask.float()

def get_transforms(encoder_name, encoder_weights, train=True):
    preprocess_fn = get_preprocessing_fn(encoder_name, pretrained=encoder_weights) if encoder_weights else None
    if train:
        return A.Compose([
            A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.8, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, p=0.5),
            A.OneOf([
                A.RandomBrightnessContrast(p=0.5),
                A.HueSaturationValue(p=0.5),
            ], p=0.5),
            A.Lambda(image=preprocess_fn) if preprocess_fn else A.NoOp(),
            ToTensorV2(transpose_mask=True)
        ])
    else:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Lambda(image=preprocess_fn) if preprocess_fn else A.NoOp(),
            ToTensorV2(transpose_mask=True)
        ])

def calculate_metrics(logits, masks):
    probs = torch.sigmoid(logits.float())
    preds = (probs > THRESHOLD).float()
    return dice_score(preds, masks).item(), iou_score(preds, masks).item()

# =========================
# ЦИКЛЫ ОБУЧЕНИЯ И ВАЛИДАЦИИ
# =========================
def train_one_epoch(model, loader, optimizer, loss_fn, scaler, ema=None):
    model.train()
    total_loss, total_dice, total_iou = 0, 0, 0
    pbar = tqdm(loader, desc="   Training", leave=False) if TQDM_AVAILABLE else loader

    for img, msk in pbar:
        img = img.to(DEVICE, dtype=torch.float32, non_blocking=True)
        msk = msk.to(DEVICE, dtype=torch.float32, non_blocking=True)

        if np.random.rand() < CUTMIX_PROB:
            img, msk = cutmix_segmentation(img, msk)

        optimizer.zero_grad(set_to_none=True)

        if scaler:
            with torch.amp.autocast('cuda'):
                logits = model(img)
                loss = loss_fn(logits, msk)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(img)
            loss = loss_fn(logits, msk)
            loss.backward()
            optimizer.step()

        if ema:
            ema.update()

        with torch.no_grad():
            d, i = calculate_metrics(logits.detach(), msk)

        total_loss += loss.item()
        total_dice += d
        total_iou += i

        if TQDM_AVAILABLE:
            pbar.set_postfix({'loss': f'{loss.item():.3f}', 'dice': f'{d:.3f}'})

    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n

@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn):
    model.eval()
    total_loss, total_dice, total_iou = 0, 0, 0
    pbar = tqdm(loader, desc="   Validating", leave=False) if TQDM_AVAILABLE else loader
    for img, msk in pbar:
        img = img.to(DEVICE, dtype=torch.float32, non_blocking=True)
        msk = msk.to(DEVICE, dtype=torch.float32, non_blocking=True)
        logits = model(img)
        loss = loss_fn(logits, msk)
        d, i = calculate_metrics(logits, msk)
        total_loss += loss.item()
        total_dice += d
        total_iou += i
    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n

# =========================
# МОДЕЛИ
# =========================
_model_cache = {}
def create_model(model_name, encoder_name):
    cache_key = f"{model_name}_{encoder_name}"
    if cache_key in _model_cache:
        return _model_cache[cache_key]
    print(f"   Создание {model_name} с {encoder_name}...")
    if model_name == "Unet":
        model = smp.Unet(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
    elif model_name == "FPN":
        model = smp.FPN(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
    elif model_name == "DeepLabV3Plus":
        model = smp.DeepLabV3Plus(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
    elif model_name == "PSPNet":
        model = smp.PSPNet(encoder_name, encoder_weights="imagenet", classes=1, activation=None)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    _model_cache[cache_key] = model
    return model

# =========================
# ОБУЧЕНИЕ ОДНОГО ФОЛДА С ЧЕКПОИНТАМИ (ИСПРАВЛЕНО)
# =========================
def train_fold(fold_idx, train_idx, val_idx, full_dataset):
    arch = FOLD_ARCHITECTURES[fold_idx]
    model_name = arch["model"]
    encoder_name = arch["encoder"]

    print(f"\n{'='*60}")
    print(f"🎯 Фолд {fold_idx+1}/{N_FOLDS} | {model_name} + {encoder_name}")
    print(f"{'='*60}")

    train_transform = get_transforms(encoder_name, "imagenet", train=True)
    val_transform   = get_transforms(encoder_name, "imagenet", train=False)

    train_ds = SegDataset(TRAIN_IMAGES, TRAIN_MASKS, transform=train_transform)
    val_ds   = SegDataset(TRAIN_IMAGES, TRAIN_MASKS, transform=val_transform)

    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(Subset(val_ds, val_idx), batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"   Батчей: train {len(train_loader)}, val {len(val_loader)}")

    model = create_model(model_name, encoder_name).to(DEVICE)
    print(f"   Устройство модели: {next(model.parameters()).device}")

    loss_fn = CombinedLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS-WARMUP_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    ema = EMAModel(model, decay=EMA_DECAY)

    # === ЧЕКПОИНТЫ ===
    checkpoint_path = SAVE_DIR / f"fold{fold_idx}_{model_name}_{encoder_name}_checkpoint.pth"
    best_path = SAVE_DIR / f"fold{fold_idx}_{model_name}_{encoder_name}_best.pth"
    
    start_epoch = 1
    best_dice = -1.0

    if checkpoint_path.exists():
        print(f"   🔄 Найден чекпоинт, загружаем...")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_dice = checkpoint.get('best_dice', -1.0)
        # Восстанавливаем EMA
        if 'ema_shadow' in checkpoint:
            ema.shadow = checkpoint['ema_shadow']
        # Восстанавливаем scaler (если есть)
        if 'scaler_state_dict' in checkpoint and scaler is not None:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        print(f"   ✅ Продолжаем с эпохи {start_epoch}, лучший Dice: {best_dice:.4f}")
    else:
        print(f"   🆕 Начинаем обучение с нуля")

    for epoch in range(start_epoch, NUM_EPOCHS+1):
        if epoch <= WARMUP_EPOCHS:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * (0.1 + 0.9 * epoch / WARMUP_EPOCHS)
        else:
            scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        train_loss, train_dice, train_iou = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, ema)
        val_loss, val_dice, val_iou = validate_one_epoch(model, val_loader, loss_fn)

        print(f"\n   Эпоха {epoch:02d} | LR: {current_lr:.2e}")
        print(f"   Train - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, IoU: {train_iou:.4f}")
        print(f"   Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")

        # Сохраняем чекпоинт после КАЖДОЙ эпохи
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'ema_shadow': ema.shadow,
        }
        if scaler is not None:
            checkpoint_data['scaler_state_dict'] = scaler.state_dict()
        torch.save(checkpoint_data, checkpoint_path)
        print(f"   💾 Чекпоинт сохранён (эпоха {epoch})")

        if val_dice > best_dice:
            best_dice = val_dice
            ema.apply_shadow()
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_dice': val_dice,
                'config': {'model_name': model_name, 'encoder_name': encoder_name,
                          'encoder_weights': 'imagenet', 'img_size': IMG_SIZE}
            }, best_path)
            ema.restore()
            print(f"   🏆 Лучшая модель обновлена (Dice: {val_dice:.4f})")

    # Удаляем чекпоинт после успешного завершения фолда
    if checkpoint_path.exists():
        checkpoint_path.unlink()
        print(f"   🧹 Чекпоинт удалён")

    return best_dice, best_path

# =========================
# 5 АРХИТЕКТУР
# =========================
FOLD_ARCHITECTURES = [
    {"model": "DeepLabV3Plus", "encoder": "timm-efficientnet-b3"},
    {"model": "DeepLabV3Plus", "encoder": "timm-efficientnet-b3"},
    {"model": "FPN", "encoder": "efficientnet-b3"},
    {"model": "FPN", "encoder": "efficientnet-b3"},
    {"model": "DeepLabV3Plus", "encoder": "timm-efficientnet-b3"},
]

# =========================
# MAIN (СЛУЧАЙНОЕ РАЗБИЕНИЕ)
# =========================
print("\n🚀 Старт 5-фолдовой кросс-валидации с улучшениями (EMA, CutMix, Combined Loss, чекпоинты)")
print("=" * 70)

full_dataset = SegDataset(TRAIN_IMAGES, TRAIN_MASKS, transform=None)
indices = np.arange(len(full_dataset))

# СЛУЧАЙНОЕ РАЗБИЕНИЕ (каждый запуск новое)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=None)

results = []
fold_dices = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices)):
    best_dice, model_path = train_fold(fold_idx, train_idx, val_idx, full_dataset)
    fold_dices.append(best_dice)
    arch = FOLD_ARCHITECTURES[fold_idx]
    results.append({'fold': fold_idx, 'model': arch['model'], 'encoder': arch['encoder'], 'val_dice': best_dice})
    print(f"\n✅ Фолд {fold_idx} завершён. Лучший Dice: {best_dice:.4f}")

print("\n" + "=" * 70)
print("🏁 ИТОГИ")
print("=" * 70)
for r in results:
    print(f"Фолд {r['fold']}: {r['model']:15s} + {r['encoder']:20s} -> Dice: {r['val_dice']:.4f}")
print(f"\n📊 Средний Dice: {np.mean(fold_dices):.4f} ± {np.std(fold_dices):.4f}")

pd.DataFrame(results).to_csv(SAVE_DIR / "cv_results_improved.csv", index=False)
print(f"\n💾 Результаты сохранены в {SAVE_DIR / 'cv_results_improved.csv'}")

🌐 Предзагрузка энкодеров...
✅ Энкодеры готовы!


🚀 Старт 5-фолдовой кросс-валидации с улучшениями (EMA, CutMix, Combined Loss, чекпоинты)
Найдено 2350 пар изображение-маска

🎯 Фолд 1/5 | DeepLabV3Plus + timm-efficientnet-b3
Найдено 2350 пар изображение-маска
Найдено 2350 пар изображение-маска
   Батчей: train 188, val 47
   Создание DeepLabV3Plus с timm-efficientnet-b3...
   Устройство модели: cuda:0
   🆕 Начинаем обучение с нуля



   Эпоха 01 | LR: 1.60e-04
   Train - Loss: 0.1662, Dice: 0.6780, IoU: 0.5583
   Val   - Loss: 0.0957, Dice: 0.8044, IoU: 0.7042
   💾 Чекпоинт сохранён (эпоха 1)
   🏆 Лучшая модель обновлена (Dice: 0.8044)



   Эпоха 02 | LR: 2.20e-04
   Train - Loss: 0.0832, Dice: 0.7794, IoU: 0.6805
   Val   - Loss: 0.0641, Dice: 0.8443, IoU: 0.7561
   💾 Чекпоинт сохранён (эпоха 2)
   🏆 Лучшая модель обновлена (Dice: 0.8443)



   Эпоха 03 | LR: 2.80e-04
   Train - Loss: 0.0638, Dice: 0.8216, IoU: 0.7335
   Val   - Loss: 0.0550, Dice: 0.8551, IoU: 0.7791
   💾 Чекпоинт сохранён (эпоха 3)
   🏆 Лучшая модель обновлена (Dice: 0.8551)



   Эпоха 04 | LR: 3.40e-04
   Train - Loss: 0.0582, Dice: 0.8331, IoU: 0.7507
   Val   - Loss: 0.0509, Dice: 0.8699, IoU: 0.7955
   💾 Чекпоинт сохранён (эпоха 4)
   🏆 Лучшая модель обновлена (Dice: 0.8699)


...


   Эпоха 36 | LR: 6.28e-05
   Train - Loss: 0.0240, Dice: 0.9304, IoU: 0.8854
   Val   - Loss: 0.0256, Dice: 0.9312, IoU: 0.8842
   💾 Чекпоинт сохранён (эпоха 36)



   Эпоха 37 | LR: 3.61e-05
   Train - Loss: 0.0236, Dice: 0.9347, IoU: 0.8895
   Val   - Loss: 0.0255, Dice: 0.9322, IoU: 0.8851
   💾 Чекпоинт сохранён (эпоха 37)



   Эпоха 38 | LR: 1.67e-05
   Train - Loss: 0.0245, Dice: 0.9305, IoU: 0.8847
   Val   - Loss: 0.0255, Dice: 0.9321, IoU: 0.8851
   💾 Чекпоинт сохранён (эпоха 38)



   Эпоха 39 | LR: 4.94e-06
   Train - Loss: 0.0238, Dice: 0.9320, IoU: 0.8875
   Val   - Loss: 0.0255, Dice: 0.9322, IoU: 0.8853
   💾 Чекпоинт сохранён (эпоха 39)



   Эпоха 40 | LR: 1.00e-06
   Train - Loss: 0.0236, Dice: 0.9355, IoU: 0.8906
   Val   - Loss: 0.0255, Dice: 0.9325, IoU: 0.8853
   💾 Чекпоинт сохранён (эпоха 40)
   🧹 Чекпоинт удалён

✅ Фолд 4 завершён. Лучший Dice: 0.9480

🏁 ИТОГИ
Фолд 0: DeepLabV3Plus   + timm-efficientnet-b3 -> Dice: 0.9070
Фолд 1: DeepLabV3Plus   + timm-efficientnet-b3 -> Dice: 0.9445
Фолд 2: FPN             + efficientnet-b3      -> Dice: 0.8818
Фолд 3: FPN             + efficientnet-b3      -> Dice: 0.9412
Фолд 4: DeepLabV3Plus   + timm-efficientnet-b3 -> Dice: 0.9480

📊 Средний Dice: 0.9245 ± 0.0259

💾 Результаты сохранены в D:\lab3 product segmentation\runs_improved\cv_results_improved.csv


In [1]:
import json
from pathlib import Path
import time

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

# =========================
# КОНФИГУРАЦИЯ
# =========================
BASE_DIR = Path(r"D:\lab3 product segmentation")
TEST_IMAGES = BASE_DIR / "test_images"
RUNS_DIR    = BASE_DIR / "runs_improved"      # папка с обученными моделями
OUTPUT_CSV  = BASE_DIR / "submission_improved_ensemble.csv"

THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

# =========================
# РАСШИРЕННЫЙ TTA (8 ТРАНСФОРМАЦИЙ)
# =========================
def apply_tta_extended(image, tta_idx):
    """Применяет TTA и возвращает transformed, swapped, tta_idx, extra_info."""
    if tta_idx == 0:   # оригинал
        return image, False, 0, None
    elif tta_idx == 1: # горизонтальный флип
        return cv2.flip(image, 1), False, 1, None
    elif tta_idx == 2: # вертикальный флип
        return cv2.flip(image, 0), False, 2, None
    elif tta_idx == 3: # поворот 90 по часовой
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE), True, 3, None
    elif tta_idx == 4: # поворот 90 против часовой
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE), True, 4, None
    elif tta_idx == 5: # поворот 180
        return cv2.rotate(image, cv2.ROTATE_180), False, 5, None
    elif tta_idx == 6: # яркость +20%
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        hsv[:,:,2] = np.clip(hsv[:,:,2] * 1.2, 0, 255).astype(np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), False, 6, None
    elif tta_idx == 7: # контраст +20%
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        lab[:,:,0] = np.clip(lab[:,:,0] * 1.2, 0, 255).astype(np.uint8)
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), False, 7, None
    else:
        return image, False, tta_idx, None

def invert_tta_extended(mask, tta_idx, original_shape, swapped, extra_info=None):
    H_orig, W_orig = original_shape
    if tta_idx == 0:
        return mask
    elif tta_idx == 1:
        return cv2.flip(mask, 1)
    elif tta_idx == 2:
        return cv2.flip(mask, 0)
    elif tta_idx == 3:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif tta_idx == 4:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif tta_idx == 5:
        return cv2.rotate(mask, cv2.ROTATE_180)
    elif tta_idx in [6, 7]:
        # цветовые преобразования не меняют геометрию маски
        return mask
    return mask

# =========================
# ПОСТОБРАБОТКА
# =========================
def postprocess(mask, min_area=100, fill_holes=True):
    mask = mask.astype(np.uint8)
    nlabels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for i in range(1, nlabels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == i] = 1
    if fill_holes:
        contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            cv2.drawContours(cleaned, [cnt], -1, 1, thickness=cv2.FILLED)
    return cleaned

def serialize_mask(mask2d):
    return json.dumps(mask2d.tolist(), separators=(',', ':'))

# =========================
# ЗАГРУЗКА МОДЕЛЕЙ С ВЕСАМИ (val_dice)
# =========================
def create_model(model_name, encoder_name):
    if model_name == "Unet":
        return smp.Unet(encoder_name, encoder_weights=None, classes=1, activation=None)
    elif model_name == "UnetPlusPlus":
        return smp.UnetPlusPlus(encoder_name, encoder_weights=None, classes=1, activation=None)
    elif model_name == "FPN":
        return smp.FPN(encoder_name, encoder_weights=None, classes=1, activation=None)
    elif model_name == "DeepLabV3Plus":
        return smp.DeepLabV3Plus(encoder_name, encoder_weights=None, classes=1, activation=None)
    elif model_name == "PSPNet":
        return smp.PSPNet(encoder_name, encoder_weights=None, classes=1, activation=None)
    else:
        raise ValueError(f"Unknown model: {model_name}")

def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    cfg = ckpt['config']
    model_name = cfg['model_name']
    encoder_name = cfg['encoder_name']
    val_dice = ckpt.get('val_dice', 0.0)
    model = create_model(model_name, encoder_name)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE)
    model.eval()
    preproc = get_preprocessing_fn(encoder_name, pretrained=cfg.get('encoder_weights'))
    return model, preproc, cfg['img_size'], val_dice

# Ищем все лучшие модели (можно также использовать SWA)
model_paths = sorted(list(RUNS_DIR.glob("fold*_best.pth")))
if not model_paths:
    model_paths = sorted(list(RUNS_DIR.glob("fold*_swa.pth")))
if not model_paths:
    raise FileNotFoundError(f"В папке {RUNS_DIR} нет файлов моделей!")

print(f"🎯 Найдено {len(model_paths)} моделей для ансамбля:")
models = []
weights = []
for p in model_paths:
    model, preproc, size, val_dice = load_model(p)
    models.append((model, preproc, size))
    weights.append(val_dice)
    print(f"  📦 {p.name} (val_dice = {val_dice:.4f})")

# Нормируем веса
weights = np.array(weights)
weights = weights / weights.sum() if weights.sum() > 0 else np.ones(len(weights))/len(weights)

# =========================
# ИНФЕРЕНС
# =========================
test_paths = sorted([p for p in TEST_IMAGES.rglob("*") if p.suffix.lower() in IMG_EXTS])
print(f"\n🖼️ Обработка {len(test_paths)} тестовых изображений...")

N_TTA = 8   # число TTA-трансформаций
rows = []
start_time = time.time()

with torch.no_grad():
    for idx, img_path in enumerate(test_paths, 1):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        H_orig, W_orig = img_rgb.shape[:2]

        ensemble_prob = np.zeros((H_orig, W_orig), dtype=np.float32)

        for (model, preproc_fn, model_size), weight in zip(models, weights):
            model_prob = np.zeros((H_orig, W_orig), dtype=np.float32)
            for tta_idx in range(N_TTA):
                transformed, swapped, _, extra = apply_tta_extended(img_rgb, tta_idx)
                inp = cv2.resize(transformed, (model_size, model_size))
                inp = inp.astype(np.float32)
                if preproc_fn:
                    inp = preproc_fn(inp)
                else:
                    inp /= 255.0
                tensor = torch.from_numpy(inp.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
                logit = model(tensor)
                prob = torch.sigmoid(logit).cpu().numpy()[0, 0]

                H_trans, W_trans = transformed.shape[:2]
                prob_resized = cv2.resize(prob, (W_trans, H_trans), interpolation=cv2.INTER_LINEAR)
                prob_restored = invert_tta_extended(prob_resized, tta_idx, (H_orig, W_orig), swapped, extra)
                model_prob += prob_restored

            model_prob /= N_TTA
            ensemble_prob += model_prob * weight

        ensemble_prob /= weights.sum()
        binary_mask = (ensemble_prob > THRESHOLD).astype(np.uint8)
        binary_mask = postprocess(binary_mask, min_area=150, fill_holes=True)

        rows.append({'ImageId': img_path.name, 'mask': serialize_mask(binary_mask)})

        if idx % 50 == 0:
            elapsed = time.time() - start_time
            speed = idx / elapsed
            remaining = (len(test_paths) - idx) / speed
            print(f"✅ Обработано {idx}/{len(test_paths)} | Скорость: {speed:.2f} img/s | Осталось: {remaining/60:.1f} мин")

# =========================
# СОХРАНЕНИЕ
# =========================
sub_df = pd.DataFrame(rows)
sub_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
elapsed_total = time.time() - start_time
print(f"\n🎉 Готово! Обработано {len(test_paths)} изображений за {elapsed_total/60:.1f} мин")
print(f"📊 Сабмит сохранён: {OUTPUT_CSV}")
print(f"📊 Взвешенный ансамбль из {len(models)} моделей")
print("\nПревью:")
print(sub_df.head())

c:\Users\vlad-\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎯 Найдено 5 моделей для ансамбля:
  📦 fold0_DeepLabV3Plus_timm-efficientnet-b3_best.pth (val_dice = 0.9070)
  📦 fold1_DeepLabV3Plus_timm-efficientnet-b3_best.pth (val_dice = 0.9445)
  📦 fold2_FPN_efficientnet-b3_best.pth (val_dice = 0.8818)
  📦 fold3_FPN_efficientnet-b3_best.pth (val_dice = 0.9412)
  📦 fold4_DeepLabV3Plus_timm-efficientnet-b3_best.pth (val_dice = 0.9480)

🖼️ Обработка 2000 тестовых изображений...
✅ Обработано 50/2000 | Скорость: 1.03 img/s | Осталось: 31.6 мин
✅ Обработано 100/2000 | Скорость: 1.04 img/s | Осталось: 30.5 мин
✅ Обработано 150/2000 | Скорость: 1.03 img/s | Осталось: 30.0 мин
✅ Обработано 200/2000 | Скорость: 1.01 img/s | Осталось: 29.7 мин
✅ Обработано 250/2000 | Скорость: 1.01 img/s | Осталось: 29.0 мин
✅ Обработано 300/2000 | Скорость: 1.00 img/s | Осталось: 28.3 мин
✅ Обработано 350/2000 | Скорость: 1.00 img/s | Осталось: 27.5 мин
✅ Обработано 400/2000 | Скорость: 1.00 img/s | Осталось: 26.6 мин
✅ Обработано 450/2000 | Скорость: 1.00 img/s | Осталось: